In [1]:
import os
import rasterio
from rasterio.warp import reproject, Resampling
from tqdm import tqdm


In [2]:

# ==========================================
# 1. PATHS
# ==========================================
MASTER_FILE = r"E:\DownloadData\ranh_gioi\LandCoverMap\lc_vn_500m_32648_rice_binary.tif"

INPUT_ROOT = r"E:\DownloadData\co2_ban_do\output_500m"          # root chứa dữ liệu 500m
OUTPUT_ROOT = r"E:\DownloadData\co2_ban_do\output_500m_align"  # folder mới để lưu khi align
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ⚠ CHỈ XỬ LÝ CÁC FOLDER NÀY
TARGET_FOLDERS = ["ndvi_evi", "optical_depth", "par", "smap"]


# ==========================================
# 2. LOAD MASTER GRID
# ==========================================
with rasterio.open(MASTER_FILE) as master:
    MASTER_TRANSFORM = master.transform
    MASTER_CRS = master.crs
    MASTER_WIDTH = master.width
    MASTER_HEIGHT = master.height
    MASTER_META = master.meta.copy()

print("MASTER GRID LOADED:")
print(" - CRS:", MASTER_CRS)
print(" - Size:", MASTER_WIDTH, "x", MASTER_HEIGHT)
print(" - Transform:", MASTER_TRANSFORM)


# ==========================================
# 3. HÀM ALIGN – chạy cho mọi raster đa band
# ==========================================
def align_to_master(src_path, dst_path):
    with rasterio.open(src_path) as src:

        meta = src.meta.copy()
        meta.update({
            "crs": MASTER_CRS,
            "transform": MASTER_TRANSFORM,
            "width": MASTER_WIDTH,
            "height": MASTER_HEIGHT
        })

        with rasterio.open(dst_path, "w", **meta) as dst:
            for b in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, b),
                    destination=rasterio.band(dst, b),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=MASTER_TRANSFORM,
                    dst_crs=MASTER_CRS,
                    resampling=Resampling.bilinear     # dùng bilinear cho dữ liệu climate
                )


# ==========================================
# 4. LOOP CHỈ Ở CÁC DATASET ĐƯỢC CHỈ ĐỊNH
# ==========================================
for dataset in TARGET_FOLDERS:

    src_dataset_path = os.path.join(INPUT_ROOT, dataset)
    if not os.path.exists(src_dataset_path):
        print(f"⚠ Bỏ qua {dataset} (không tồn tại)")
        continue

    dst_dataset_path = os.path.join(OUTPUT_ROOT, dataset)
    os.makedirs(dst_dataset_path, exist_ok=True)

    print(f"\n🚀 ALIGNING DATASET: {dataset}")

    # duyệt các folder theo năm
    year_folders = os.listdir(src_dataset_path)

    for yfolder in year_folders:

        src_year_path = os.path.join(src_dataset_path, yfolder)
        if not os.path.isdir(src_year_path):
            continue

        dst_year_path = os.path.join(dst_dataset_path, yfolder)
        os.makedirs(dst_year_path, exist_ok=True)

        tif_files = [f for f in os.listdir(src_year_path) if f.endswith(".tif")]

        for f in tqdm(tif_files, desc=f"{dataset}/{yfolder}"):
            src_file = os.path.join(src_year_path, f)
            dst_file = os.path.join(dst_year_path, f.replace(".tif", "_aligned.tif"))

            align_to_master(src_file, dst_file)

print("\n🎉 HOÀN THÀNH — TẤT CẢ DỮ LIỆU ĐÃ ĐƯỢC ALIGN GRID CHUẨN!")


MASTER GRID LOADED:
 - CRS: EPSG:32648
 - Size: 1613 x 3335
 - Transform: | 500.00, 0.00, 185512.76|
| 0.00,-500.00, 2594069.36|
| 0.00, 0.00, 1.00|

🚀 ALIGNING DATASET: ndvi_evi


ndvi_evi/ndvi_evi_2024_500m: 100%|██████████| 31/31 [01:50<00:00,  3.55s/it]



🚀 ALIGNING DATASET: optical_depth


optical_depth/aod_2024: 100%|██████████| 31/31 [01:00<00:00,  1.94s/it]



🚀 ALIGNING DATASET: par


par/par_2024: 100%|██████████| 24/24 [01:38<00:00,  4.09s/it]



🚀 ALIGNING DATASET: smap


smap/smap_2024: 100%|██████████| 31/31 [00:38<00:00,  1.24s/it]


🎉 HOÀN THÀNH — TẤT CẢ DỮ LIỆU ĐÃ ĐƯỢC ALIGN GRID CHUẨN!


In [1]:
import os
import rasterio
from rasterio.warp import reproject, Resampling
from tqdm import tqdm

# ==========================================
# 1. PATHS
# ==========================================
MASTER_FILE = r"E:\DownloadData\ranh_gioi\LandCoverMap\lc_vn_500m_32648_rice_binary.tif"

# Folder chứa TẤT CẢ file TIFF cần align
INPUT_FOLDER = r"E:\DownloadData\co2_ban_do\output_500m_soilgrids"

# Folder để lưu TIFF sau khi align
OUTPUT_FOLDER = r"E:\DownloadData\co2_ban_do\output_500m_align_soilgrids"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# ==========================================
# 2. LOAD MASTER GRID
# ==========================================
with rasterio.open(MASTER_FILE) as master:
    MASTER_TRANSFORM = master.transform
    MASTER_CRS = master.crs
    MASTER_WIDTH = master.width
    MASTER_HEIGHT = master.height
    MASTER_META = master.meta.copy()

print("MASTER GRID LOADED:")
print(" - CRS:", MASTER_CRS)
print(" - Size:", MASTER_WIDTH, "x", MASTER_HEIGHT)
print(" - Transform:", MASTER_TRANSFORM)


# ==========================================
# 3. HÀM ALIGN – HỖ TRỢ MULTI-BAND TIFF
# ==========================================
def align_to_master(src_path, dst_path):
    with rasterio.open(src_path) as src:

        meta = src.meta.copy()
        meta.update({
            "crs": MASTER_CRS,
            "transform": MASTER_TRANSFORM,
            "width": MASTER_WIDTH,
            "height": MASTER_HEIGHT
        })

        with rasterio.open(dst_path, "w", **meta) as dst:
            # xử lý mọi bands
            for b in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, b),
                    destination=rasterio.band(dst, b),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=MASTER_TRANSFORM,
                    dst_crs=MASTER_CRS,
                    resampling=Resampling.bilinear
                )


# ==========================================
# 4. ALIGN TẤT CẢ TIFF TRONG 1 FOLDER
# ==========================================
tif_files = sorted([f for f in os.listdir(INPUT_FOLDER) if f.endswith(".tif")])

print(f"\n🚀 Đang align {len(tif_files)} file trong thư mục: {INPUT_FOLDER}")

for f in tqdm(tif_files, desc="Aligning files"):
    src_file = os.path.join(INPUT_FOLDER, f)
    dst_file = os.path.join(OUTPUT_FOLDER, f.replace(".tif", "_aligned.tif"))
    align_to_master(src_file, dst_file)

print("\n🎉 ĐÃ HOÀN THÀNH — TOÀN BỘ TIFF TRONG FOLDER ĐÃ ALIGN XONG!")


MASTER GRID LOADED:
 - CRS: EPSG:32648
 - Size: 1613 x 3335
 - Transform: | 500.00, 0.00, 185512.76|
| 0.00,-500.00, 2594069.36|
| 0.00, 0.00, 1.00|

🚀 Đang align 6 file trong thư mục: E:\DownloadData\co2_ban_do\output_500m_soilgrids


Aligning files: 100%|██████████| 6/6 [00:02<00:00,  2.38it/s]


🎉 ĐÃ HOÀN THÀNH — TOÀN BỘ TIFF TRONG FOLDER ĐÃ ALIGN XONG!
